In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ===========================
# 1️⃣ Imports
# ===========================
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# ===========================
# 2️⃣ Device
# ===========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
# ===========================
# 2️⃣ Load Dataset
# ===========================
from datasets import load_dataset

ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
print(ds)

# نحوّل لـ pandas
train_df = ds["train"].to_pandas()
test_df  = ds["validation"].to_pandas()

train_df = train_df.dropna().reset_index(drop=True)
test_df  = test_df.dropna().reset_index(drop=True)

print(f"\nTrain size: {train_df.shape}")
print(f"Test size:  {test_df.shape}")
print(f"Label distribution (train):\n{train_df['label'].value_counts()}")
print(f"Label distribution (test):\n{test_df['label'].value_counts()}")

Using device: cuda
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})

Train size: (9543, 2)
Test size:  (2388, 2)
Label distribution (train):
label
2    6178
1    1923
0    1442
Name: count, dtype: int64
Label distribution (test):
label
2    1566
1     475
0     347
Name: count, dtype: int64


In [ ]:

model_name="Qwen/Qwen3-0.6B"
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Verify GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computing Device: {device}")

Computing Device: cuda


# Data Preprocessing and Tokenization

In [ ]:


# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure padding token is set to EOS token if not defined
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Define tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Apply tokenization to the dataset
tokenized_ds = ds.map(tokenize_function, batched=True)

# Prepare columns for training
tokenized_ds = tokenized_ds.remove_columns(["text"])
tokenized_ds = tokenized_ds.rename_column("label", "labels")
tokenized_ds = tokenized_ds.cast_column("labels", Value('int64')) # Changed to datasets.Value('int64') for classification labels
tokenized_ds.set_format("torch")

print("Data preprocessing completed successfully.")

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/2388 [00:00<?, ? examples/s]

Data preprocessing completed successfully.


# Model Loading and LoRA Configuration

In [ ]:
# Just for zero shot
# Load the base model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3, # 3 labels
    device_map="auto",
    pad_token_id=tokenizer.pad_token_id # Explicitly set pad_token_id for the model
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Load the base model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3, # 3 labels
    device_map="auto",
    pad_token_id=tokenizer.pad_token_id # Explicitly set pad_token_id for the model
)

# Configure LoRA parameters specific to GPT-2 architecture
# GPT-2 uses Conv1D layers named c_attn, c_proj, c_fc
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
   # target_modules=["gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS"
)
# Configure LoRA parameters specific to GPT-2 architecture
# GPT-2 uses Conv1D layers named c_attn, c_proj, c_fc

# Apply LoRA configuration to the model
model = get_peft_model(model, lora_config)

# Display trainable parameters
print("\n--- LoRA Configuration Summary ---")
model.print_trainable_parameters()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- LoRA Configuration Summary ---
trainable params: 10,095,616 || all params: 606,148,608 || trainable%: 1.6655


# Training Arguments and Metrics Definition

In [ ]:
# Define evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# Define training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Qwn3-rotten-tomatoes/Qwn3_sentiment_finetuned",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch", # Changed from evaluation_strategy to eval_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    logging_steps=10,
    fp16=False # Changed to False to avoid BFloat16 NotImplementedError
)

# Initialize Data Collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Traing the model

In [ ]:
#Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Execute the training process
print("Initiating training process...")
trainer.train()
print("Training completed.")

Initiating training process...
Training completed.


# Model Evaluation

In [ ]:
# Perform final evaluation on the test set
print("\n--- Final Evaluation on Test Set ---")
predictions = trainer.predict(tokenized_ds["validation"])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Compute final metrics
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')

# Display results
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")


--- Final Evaluation on Test Set ---


Accuracy:  0.3304
Precision: 0.4616
Recall:    0.3304
F1-Score:  0.3658
